# EPCAPE data availability: % of each month with data

The *DataAvailability* tab of `Table-EPCAPE_SeasonalAverages.xlsx` gives, for each instrument, the fraction of each campaign month (Feb 2023 – Feb 2024) with data. Most instruments are already filled in; this notebook computes the same thing for the ones that are not, starting with the **Fog Droplet Monitor (DMT FM-120)**.

**Data.** `data/processed/FM120/Monthly FM-120 Files/` holds one CSV per month (`YYYY-MM.csv`) of **5-minute averages**. Each row is one 5-min sample; times with no data simply have no row.

**Method: total time with data.** Each valid 5-min sample stands for 5 minutes of measurement, so

$$\text{availability}_{\text{month}} = \frac{N_{\text{samples}} \times 5\ \text{min}}{\text{total time in the month}}$$

For comparison, the notebook also gives the fraction of *days* with at least one sample. That number is higher, because a day with a single 5-min sample counts as a full day.

**Denominator.** EPCAPE ran 2023-02-15 to 2024-02-14 (`config.yaml`, `campaign:`). The ARM rows of the tab show 100 % for Feb 2023, so they appear to count only the deployment days in the first and last months. `USE_CAMPAIGN_WINDOW` below switches between that and whole calendar months.

In [1]:
# --- Imports ------------------------------------------------------------------
from pathlib import Path

import pandas as pd
import yaml

In [2]:
# --- Settings -----------------------------------------------------------------
# The notebook lives in EPCAPE/comparisons/data_availability/, so the EPCAPE folder is two levels up.
EPCAPE_DIR = Path.cwd().parents[1]
FM120_DIR = EPCAPE_DIR / "data" / "processed" / "FM120" / "Monthly FM-120 Files"

# Length of one FM-120 averaging interval: each row represents 5 minutes of measurement.
SAMPLE_INTERVAL = pd.Timedelta(minutes=5)

# Campaign window, read from config.yaml so it matches the rest of the EPCAPE code.
with open(EPCAPE_DIR / "config.yaml") as f:
    campaign = yaml.safe_load(f)["campaign"]
CAMPAIGN_START = pd.Timestamp(campaign["start"])                       # 2023-02-15 00:00 (inclusive)
CAMPAIGN_END = pd.Timestamp(campaign["end"]) + pd.Timedelta(days=1)   # 2024-02-15 00:00 (exclusive)

# True  -> the first/last months only count time inside the campaign window (what the ARM rows seem to do).
# False -> every month uses its full calendar length.
USE_CAMPAIGN_WINDOW = True

print("FM-120 folder:", FM120_DIR)
print("Campaign:", CAMPAIGN_START, "to", CAMPAIGN_END, "(end exclusive)")

FM-120 folder: /Users/andrewbuggee/Documents/VS_CODE/Python-Research/EPCAPE/data/processed/FM120/Monthly FM-120 Files
Campaign: 2023-02-15 00:00:00 to 2024-02-15 00:00:00 (end exclusive)


## 1. Read the monthly 5-min files into one time series

* The first column of each CSV is the timestamp; all months are stacked into one table indexed by time.
* A sample counts as **valid** if any of the bulk quantities (temperature, airspeed, number concentration, LWC, MVD, ED) is present. In these files every row has all of them, so this only guards against future gaps.
* Samples outside the campaign window are dropped (the Feb 2024 file runs to 2024-02-15 17:00).
* Times are UTC, and each timestamp marks the start of its 5-min interval (`data/processed/readme_FM120.md`: files run 00:00 to 23:55 UTC).

In [3]:
# --- Load and stack all monthly files -----------------------------------------
files = sorted(FM120_DIR.glob("*.csv"))
fm120 = pd.concat([pd.read_csv(f, index_col=0, parse_dates=True) for f in files]).sort_index()

# Each 5-min interval should appear once and sit on the 5-min grid; stop if not,
# since the "N_samples x 5 min" method below relies on both.
assert not fm120.index.duplicated().any(), "duplicate timestamps"
assert (fm120.index == fm120.index.floor(SAMPLE_INTERVAL)).all(), "timestamps off the 5-min grid"

# Keep samples where at least one bulk quantity was measured.
BULK_COLUMNS = ["T Ambient (C)", "PAS (m/s)", "Number Conc (#/cm^3)", "LWC (g/m^3)", "MVD (um)", "ED (um)"]
fm120 = fm120.dropna(subset=BULK_COLUMNS, how="all")

# Keep only samples inside the campaign window.
in_campaign = (fm120.index >= CAMPAIGN_START) & (fm120.index < CAMPAIGN_END)
print(f"Samples outside the campaign window (dropped): {(~in_campaign).sum()}")
fm120 = fm120[in_campaign]

print(f"{len(files)} files, {len(fm120)} valid 5-min samples, "
      f"{fm120.index.min()} to {fm120.index.max()}")

Samples outside the campaign window (dropped): 205
13 files, 89906 valid 5-min samples, 2023-02-16 16:40:00 to 2024-02-14 23:55:00


## 2. Time with data vs. total time in each month

In [4]:
# --- Numerator: time with data in each month ---------------------------------
month = fm120.index.to_period("M")
samples = fm120.groupby(month).size()
hours_with_data = samples * SAMPLE_INTERVAL / pd.Timedelta(hours=1)

# Also count days with at least one sample, for comparison with the time-based number.
days_with_data = pd.Series(fm120.index.normalize().unique()).dt.to_period("M").value_counts()

# --- Denominator: total time each month could have been measured -------------
# Build every hour that is eligible (campaign window, or whole calendar months spanning it)
# and count how many fall in each month.
if USE_CAMPAIGN_WINDOW:
    period_start, period_end = CAMPAIGN_START, CAMPAIGN_END
else:
    period_start = CAMPAIGN_START.to_period("M").start_time
    period_end = (CAMPAIGN_END - pd.Timedelta(days=1)).to_period("M").end_time.ceil("D")
all_days = pd.date_range(period_start, period_end, freq="D", inclusive="left")
possible_days = pd.Series(1, index=all_days).groupby(all_days.to_period("M")).size()
possible_hours = possible_days * 24

# --- Combine into one table --------------------------------------------------
avail = pd.DataFrame({
    "possible_hours": possible_hours,
    "hours_with_data": hours_with_data,
    "possible_days": possible_days,
    "days_with_data": days_with_data,
}).fillna(0)
avail["time_fraction"] = avail["hours_with_data"] / avail["possible_hours"]
avail["day_fraction"] = avail["days_with_data"] / avail["possible_days"]
avail.index.name = "month"
avail.round(3)

,possible_hours,hours_with_data,possible_days,days_with_data,time_fraction,day_fraction
month,,,,,,
2023-02,336,81.917,14,9,0.244,0.643
2023-03,744,669.167,31,30,0.899,0.968
2023-04,720,601.417,30,28,0.835,0.933
2023-05,744,591.833,31,28,0.795,0.903
2023-06,720,711.667,30,30,0.988,1.000
2023-07,744,735.583,31,31,0.989,1.000
2023-08,744,681.667,31,31,0.916,1.000
2023-09,720,705.333,30,30,0.980,1.000
2023-10,744,551.500,31,26,0.741,0.839


## 3. Result in the layout of the DataAvailability tab

One row per method, one column per month, as a percentage (the lower block of the sheet uses percent with one decimal). The **time-based row is the one to use for the table**; the day-based row is shown for reference.

In [5]:
# --- Sheet-style summary (percent, months as columns) -------------------------
summary = (avail[["time_fraction", "day_fraction"]].T * 100).round(1)
summary.index = ["FM-120: % of time with data", "FM-120: % of days with data"]
summary.columns = [m.strftime("%b %Y") for m in summary.columns]
summary

,Feb 2023,Mar 2023,Apr 2023,May 2023,Jun 2023,Jul 2023,Aug 2023,Sep 2023,Oct 2023,Nov 2023,Dec 2023,Jan 2024,Feb 2024
FM-120: % of time with data,24.4,89.9,83.5,79.5,98.8,98.9,91.6,98.0,74.1,98.8,90.8,58.9,100.0
FM-120: % of days with data,64.3,96.8,93.3,90.3,100.0,100.0,100.0,100.0,83.9,100.0,93.5,80.6,100.0


## 4. Periods when the instrument was not recording

A **gap** is a stretch of time with no 5-min sample. Each timestamp marks the *start* of its 5-min interval, so a sample at time *t* covers *t* to *t* + 5 min. Between two consecutive samples the gap therefore runs from (previous sample + 5 min) to (next sample), and adjacent samples (5 min apart) have no gap.

The stretches between the campaign start and the first sample, and between the last sample and the campaign end, count as gaps too.

FM-120 rows exist in clear air too (78 % of samples have N < 1 cm⁻³ and 88 % have LWC < 0.001 g m⁻³), and visibility at Mt. Soledad is distributed the same way inside and outside the gaps (visibility < 1 km 6.6 % vs 7.2 % of the time). So a gap means the instrument was not producing data, not that there was no cloud or haze.

In [6]:
# --- Find every gap -----------------------------------------------------------
# Start and end of every 5-min interval that has data.
interval_start = fm120.index.to_series()
interval_end = interval_start + SAMPLE_INTERVAL

# A gap begins where one interval ends and the next one does not start right away.
# Add the campaign start/end as boundaries, so that missing data at either end is counted.
gap_start = pd.concat([pd.Series([CAMPAIGN_START]), interval_end]).reset_index(drop=True)
gap_end = pd.concat([interval_start, pd.Series([CAMPAIGN_END])]).reset_index(drop=True)
gaps = pd.DataFrame({"start": gap_start, "end": gap_end})
gaps["duration"] = gaps["end"] - gaps["start"]
gaps = gaps[gaps["duration"] > pd.Timedelta(0)].reset_index(drop=True)

# Split into long gaps (1 day or longer, listed individually) and the rest (summarised).
LONG_GAP = pd.Timedelta(days=1)
long_gaps = gaps[gaps["duration"] >= LONG_GAP]
short_gaps = gaps[gaps["duration"] < LONG_GAP]

# Consistency check: time with data + time in gaps must equal the whole campaign window.
assert len(fm120) * SAMPLE_INTERVAL + gaps["duration"].sum() == CAMPAIGN_END - CAMPAIGN_START

print(f"{len(gaps)} gaps, {gaps['duration'].sum() / pd.Timedelta(hours=1):.1f} h in total")
print(f"{len(long_gaps)} of 1 day or longer, {len(short_gaps)} shorter than 1 day")
long_gaps

62 gaps, 1267.8 h in total
17 of 1 day or longer, 45 shorter than 1 day


,start,end,duration
0,2023-02-15 00:00:00,2023-02-16 16:40:00,1 days 16:40:00
2,2023-02-17 16:10:00,2023-02-20 10:40:00,2 days 18:30:00
3,2023-02-20 14:30:00,2023-02-22 12:25:00,1 days 21:55:00
4,2023-02-22 12:40:00,2023-02-24 15:15:00,2 days 02:35:00
11,2023-03-31 00:00:00,2023-04-01 00:00:00,1 days 00:00:00
15,2023-04-16 16:30:00,2023-04-19 00:00:00,2 days 07:30:00
22,2023-05-05 00:00:00,2023-05-08 01:55:00,3 days 01:55:00
25,2023-05-30 13:30:00,2023-05-31 17:55:00,1 days 04:25:00
32,2023-08-22 20:10:00,2023-08-23 20:50:00,1 days 00:40:00
38,2023-10-17 00:00:00,2023-10-19 02:35:00,2 days 02:35:00


## 5. Write the text report

Writes `FM120_data_availability.txt` next to this notebook: the monthly % of time with data, the gaps of one day or longer, and statistics on the shorter gaps.

In [7]:
# --- Helpers for readable output ----------------------------------------------
def hours(td: pd.Timedelta) -> float:
    """Length of a time span in hours."""
    return td / pd.Timedelta(hours=1)


def stamp(t: pd.Timestamp) -> str:
    """Timestamp as 'YYYY-MM-DD HH:MM'."""
    return t.strftime("%Y-%m-%d %H:%M")


# --- Build the report line by line -------------------------------------------
total_time = CAMPAIGN_END - CAMPAIGN_START
time_with_data = len(fm120) * SAMPLE_INTERVAL
lines = [
    "FM-120 Fog Droplet Monitor (Mt. Soledad), EPCAPE: data availability",
    "=" * 68,
    f"Data:     5-min averages, data/processed/FM120/Monthly FM-120 Files/ (UTC)",
    f"Campaign: {stamp(CAMPAIGN_START)} to {stamp(CAMPAIGN_END)} UTC (end exclusive)",
    f"Method:   % available = (number of 5-min samples x 5 min) / (time in the month",
    f"          inside the campaign window). Feb 2023 counts Feb 15-28 only;",
    f"          Feb 2024 counts Feb 1-14 only.",
    f"Made by:  EPCAPE/comparisons/data_availability/data_availability.ipynb",
    "",
    "1. Percent of time with data, by month",
    "-" * 40,
    f"{'Month':<10}{'Hours with data':>17}{'Hours possible':>16}{'% available':>13}",
]
for m, row in avail.iterrows():
    lines.append(f"{m.strftime('%b %Y'):<10}{row['hours_with_data']:>17.1f}"
                 f"{row['possible_hours']:>16.0f}{100 * row['time_fraction']:>13.1f}")
lines.append(f"{'Campaign':<10}{hours(time_with_data):>17.1f}{hours(total_time):>16.0f}"
             f"{100 * time_with_data / total_time:>13.1f}")

lines += [
    "",
    "2. Periods with no data of 1 day or longer",
    "-" * 40,
    "Start = end of the last 5-min interval with data; End = start of the next one.",
    f"{'Start (UTC)':<19}{'End (UTC)':<19}{'Duration (days)':>16}",
]
for _, g in long_gaps.iterrows():
    lines.append(f"{stamp(g['start']):<19}{stamp(g['end']):<19}{hours(g['duration']) / 24:>16.2f}")
lines.append(f"Total: {len(long_gaps)} periods, {hours(long_gaps['duration'].sum()) / 24:.1f} days "
             f"({100 * long_gaps['duration'].sum() / total_time:.1f}% of the campaign)")

# Statistics on the shorter gaps, plus how many fall into a few duration classes.
d_h = short_gaps["duration"].map(hours)
classes = pd.cut(d_h, bins=[0, 1, 6, 12, 24], right=False,
                 labels=["< 1 h", "1-6 h", "6-12 h", "12-24 h"])
lines += [
    "",
    "3. Periods with no data shorter than 1 day",
    "-" * 40,
    f"Number of periods:  {len(short_gaps)}",
    f"Total time:         {d_h.sum():.1f} h ({100 * short_gaps['duration'].sum() / total_time:.1f}% of the campaign)",
    f"Median duration:    {d_h.median():.1f} h",
    f"Mean duration:      {d_h.mean():.1f} h",
    f"Shortest / longest: {d_h.min():.2f} h / {d_h.max():.1f} h",
    "Number by duration:",
]
for label, n in classes.value_counts(sort=False).items():
    lines.append(f"    {label:<8} {n:>3}")

lines += [
    "",
    "Note: the FM-120 records clear air as well (most samples have N < 1 cm-3), so",
    "a period with no data means the instrument was not recording, not that there",
    "was no cloud or haze.",
]

# --- Write and show the report ------------------------------------------------
REPORT = Path.cwd() / "FM120_data_availability.txt"
REPORT.write_text("\n".join(lines) + "\n")
print(REPORT.read_text())

FM-120 Fog Droplet Monitor (Mt. Soledad), EPCAPE: data availability
Data:     5-min averages, data/processed/FM120/Monthly FM-120 Files/ (UTC)
Campaign: 2023-02-15 00:00 to 2024-02-15 00:00 UTC (end exclusive)
Method:   % available = (number of 5-min samples x 5 min) / (time in the month
          inside the campaign window). Feb 2023 counts Feb 15-28 only;
          Feb 2024 counts Feb 1-14 only.
Made by:  EPCAPE/comparisons/data_availability/data_availability.ipynb

1. Percent of time with data, by month
----------------------------------------
Month       Hours with data  Hours possible  % available
Feb 2023               81.9             336         24.4
Mar 2023              669.2             744         89.9
Apr 2023              601.4             720         83.5
May 2023              591.8             744         79.5
Jun 2023              711.7             720         98.8
Jul 2023              735.6             744         98.9
Aug 2023              681.7             744     